In [1]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

/home/tanveer/dependencies/python3/genai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [3]:
embedding_model = GoogleGenerativeAIEmbeddings(
  model="models/gemini-embedding-001"
)

vector_store = Chroma.from_documents(
  documents=documents,
  embedding=embedding_model,
  collection_name="my_collection"
)



In [7]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [8]:
query = "what is chroma used for?"

result = retriever.invoke(query)

In [9]:
for i, doc in enumerate(result):
    print(f"Result {i+1}: {doc.page_content}")

Result 1: Chroma is a vector database optimized for LLM-based search.
Result 2: LangChain helps developers build LLM applications easily.


# MMR

In [32]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [33]:
vector_store = Chroma.from_documents(
  documents=docs,
  embedding=embedding_model,
  collection_name="my_collection_1"
)

In [43]:
retriever = vector_store.as_retriever(
  search_type = "mmr",
  search_kwargs = {"k": 3, "lambda_mult": 0.5} # lambda_mult - relevance vs diversity, 0 - only relevance, 1 - only diversity
)

In [44]:
query = "what is langchain?"
results = retriever.invoke(query)

In [45]:
for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.page_content}")

Result 1: LangChain is used to build LLM based applications.
Result 2: LangChain supports Chroma, FAISS, Pinecone, and more.
Result 3: Chroma is used to store and search document embeddings.


# Multi query retriver

In [71]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

embedding_model = GoogleGenerativeAIEmbeddings(
  model="models/gemini-embedding-001"
)

GEMINI_MODEL = "gemini-2.5-flash"
# GEMINI_MODEL = "gemma-3-27b-it"

model = ChatGoogleGenerativeAI(model=GEMINI_MODEL)

In [72]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [73]:
vector_store = Chroma.from_documents(
  documents=all_docs,
  embedding=embedding_model,
  collection_name="my_collection_3"
)

In [74]:
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [75]:
multiquery_retriever = MultiQueryRetriever.from_llm(
  retriever=vector_store.as_retriever(
    search_kwargs = {"k": 5}
  ),
  llm=model
)

In [76]:
query = "How to improve energy levels and maintain balance?"

In [77]:
similarity_result = similarity_retriever.invoke(query)
multiquery_result = multiquery_retriever.invoke(query)

In [78]:
for i, doc in enumerate(similarity_result):
  print(f"Similarity result {i+1}: {doc.page_content}")

print("*"*150)

for i, doc in enumerate(multiquery_result):
  print(f"MultiQuery result {i+1}: {doc.page_content}")

Similarity result 1: Drinking sufficient water throughout the day helps maintain metabolism and energy.
Similarity result 2: The solar energy system in modern homes helps balance electricity demand.
Similarity result 3: Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Similarity result 4: Consuming leafy greens and fruits helps detox the body and improve longevity.
Similarity result 5: Regular walking boosts heart health and can reduce symptoms of depression.
******************************************************************************************************************************************************
MultiQuery result 1: Drinking sufficient water throughout the day helps maintain metabolism and energy.
MultiQuery result 2: Consuming leafy greens and fruits helps detox the body and improve longevity.
MultiQuery result 3: Regular walking boosts heart health and can reduce symptoms of depression.
MultiQuery result 4: Mindfulness and controlled breathi

# Contextual Compression Retriever

In [81]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [82]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [83]:
embedding_model = GoogleGenerativeAIEmbeddings(
  model="models/gemini-embedding-001"
)

# GEMINI_MODEL = "gemini-2.5-flash"
GEMINI_MODEL = "gemma-3-27b-it"

model = ChatGoogleGenerativeAI(model=GEMINI_MODEL)

vector_store = Chroma.from_documents(
  documents=docs,
  embedding=embedding_model,
  collection_name="my_collection_4"
)

base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [84]:
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL)
compressor = LLMChainExtractor.from_llm(llm)

In [85]:
compression_retriever = ContextualCompressionRetriever(
  base_compressor=compressor,
  base_retriever=base_retriever
)

In [86]:
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(compressed_results):
  print(f"Compressed Result {i+1}: {doc.page_content}") 

Compressed Result 1: Photosynthesis is the process by which green plants convert sunlight into energy.
Compressed Result 2: Photosynthesis does not occur in animal cells.
Compressed Result 3: The chlorophyll in plant cells captures sunlight during photosynthesis.
